In [1]:
import operator
from typing import List, Dict, Annotated, TypedDict, Any
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
import json
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_cohere import CohereEmbeddings
from langchain_community.embeddings import JinaEmbeddings
from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\krdhi\OneDrive\Desktop\RAG_tutorials\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

"""Our Embedding Model"""
try:
    embed_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
except Exception as e:
    print(f"Error Loading the Embedding model\nError: {e}")


C:\Users\krdhi\AppData\Local\Temp\ipykernel_45044\2338105487.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9596.96it/s]


In [3]:
#Global state
class GlobalState(TypedDict):
    original_query: str
    search_queries: dict
    gathered_chunks: List[Any]
    failed_urls: Annotated[List[str],operator.add]
    missing_information: List[str]
    draft_report: str
    is_complete: bool
    formatted_references: List[dict]


In [4]:
#prompts

GENERATE_SEARCH_QUERY_PROMPT = """
You are an autonomous Master Research Supervisor. 
Your objective is to analyze a user's research topic and formulate a highly optimized, multi-faceted search strategy.

For each search query you generate, you must determine the best approach: a targeted domain search OR an open web search.

### Domain Selection Guidelines:
1. Target Domains (`target_domains`): 
   - You are NOT restricted to any predefined menu. Autonomously select the most authoritative domains for the specific niche (e.g., use "nih.gov" for medicine, "github.com" for code, "reuters.com" or local news websites for news).
   - If the topic is very broad, local, or you are unsure of the best specific domains, leave this array COMPLETELY EMPTY []. An empty array triggers an Open Web Search.

2. Exclude Domains (`exclude_domains`):
   - Use this to actively block low-quality, irrelevant, or consumer-level sites that pollute research data. 
   - If performing an Open Web Search, you SHOULD use this to filter out sites like "quora.com", "pinterest.com", "reddit.com", or "wikipedia.org" unless explicitly relevant to the user's prompt.

### Output Constraints:
- You must generate 1 distinct search strategies.
- Do NOT output any conversational text, markdown formatting blocks (like ```json), or explanations.
- You MUST output ONLY a valid JSON object in this exact format:

{
  "searches": [
    {
      "query": "climate responsive architecture rural schools bihar",
      "target_domains": [], 
      "exclude_domains": ["quora.com", "pinterest.com", "wikipedia.org"]
    },
    {
      "query": "thermal mass properties of bamboo vs stabilized earth",
      "target_domains": ["arxiv.org", "researchgate.net", "sciencedirect.com"],
      "exclude_domains": []
    },
    {
      "query": "how to build agentic RAG langgraph architecture",
      "target_domains": ["medium.com", "towardsdatascience.com", "github.com"],
      "exclude_domains": ["reddit.com"]
    }
  ]
}
"""

GENERATE_MISSING_INFO_QUERY_PROMPT = """
You are an autonomous Master Research Supervisor operating in CORRECTION MODE.
A previous research draft was evaluated, and specific critical information is missing.

Your objective is to formulate a highly optimized search strategy to find ONLY the missing information. Do NOT research the entire original topic again.

### Inputs you will evaluate:
- "Original Topic": The broad context of the research.
- "Missing Information": The specific data points, statistics, or facts that must be found.

### Search Strategy & Domain Selection:
1. Laser Focus: Your queries must be hyper-specific to the missing data. Use search operators if necessary (e.g., "compressive strength" AND "bamboo").
2. Target Domains (`target_domains`): Autonomously select the most authoritative domains likely to contain this specific data (e.g., academic journals for material strength, local government sites for regional data). If the missing info is niche, leave this array COMPLETELY EMPTY [] to trigger an Open Web Search.
3. Exclude Domains (`exclude_domains`): Actively block low-quality, irrelevant, or consumer-level sites. During an Open Web Search, you SHOULD filter out sites like "quora.com", "pinterest.com", or "wikipedia.org" to ensure high data integrity.

### Output Constraints:
- Generate 2 to 3 distinct, highly targeted search strategies.
- Do NOT output any conversational text, markdown formatting blocks (like ```json), or explanations.
- You MUST output ONLY a valid JSON object in this exact format:

{
  "searches": [
    {
      "query": "compressive load testing stabilized earth blocks vs fired bricks",
      "target_domains": ["sciencedirect.com", "researchgate.net", "springer.com"],
      "exclude_domains": []
    },
    {
      "query": "local building bylaws primary schools Muzaffarpur rural site area",
      "target_domains": [],
      "exclude_domains": ["quora.com", "pinterest.com", "wikipedia.org", "reddit.com"]
    }
  ]
}"""

GENERATE_FINAL_REPORT_PROMPT = """
Based on provided verified information in this format:
    [{
        "content": content,
        "url": url
    }]

    against a given user's QUERY.

    Using ONLY the provided verified information, generate a structured
    and detailed research-style report answering the user query.

    Output rules:
    - Start with a clear, concise Title formatted as a Markdown Header (e.g., # Title).
    - Follow a proper format for report writing (using subheadings, bullet points where necessary).
    - Do NOT include extra conversational tokens (like "Here is the report"). Output ONLY the report content.
    - Do not repeat the URL for each info chunk; instead, consolidate them into a "References" section at the end.
    - If the provided information is insufficient to confidently answer any part of the query, explicitly state the missing information instead of guessing.
"""

In [5]:
# research_query:str=input('Enter the research topic')
# print(research_query)
# write a detailed report on history, cause, effect, and solution of bihar flood.

In [6]:
# # llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,api_key=GROQ_API_KEY, max_tokens=11500)
# # llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", temperature=0)

# def supervisor_agent(state:GlobalState):
#     print("=== Supervisor Agent working...")
#     original_query = state['original_query']
#     missing = state.get("missing_information", [])
#     failed_links = state.get("failed_urls", [])

#     if missing:
#         system_message = GENERATE_MISSING_INFO_QUERY_PROMPT
#         human_message = f"""
#         Original Topic: {original_query}\nMissing Information: {missing}
        
#         CRITICAL: The following URLs previously failed to load or blocked our scraper.You MUST use the 'exclude_domains' parameter to avoid these sites: {failed_links}"""

#         print(f"-> Supervisor: Executing Correction Search for: {missing}")
#     else:
#         system_message = GENERATE_SEARCH_QUERY_PROMPT
#         human_message = f"Topic: {original_query}"
#         print("-> Supervisor: Executing Initial Broad Search Strategy")

#     messages = [SystemMessage(content=system_message),
#                 HumanMessage(content=human_message)]
    
#     search_query_response = llm.invoke(messages)
#     print(search_query_response)
#     search_queries = json.loads(search_query_response.content)
#     # print(search_queries)
#     return {"search_queries": search_queries}


In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest", 
    temperature=0
)

import json
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Define the precise JSON blueprint using Pydantic
class SingleSearchStrategy(BaseModel):
    query: str = Field(description="The optimized, multi-faceted search query string")
    target_domains: List[str] = Field(default=[], description="Authoritative niche domains to search within")
    exclude_domains: List[str] = Field(default=[], description="Low quality domains to filter out")

class SupervisorSearchSchema(BaseModel):
    searches: List[SingleSearchStrategy] = Field(description="List of 1 distinct search strategies")


def supervisor_agent(state: GlobalState):
    print("=== Supervisor Agent working...")
    original_query = state['original_query']
    missing = state.get("missing_information", [])
    failed_links = state.get("failed_urls", [])

    if missing:
        system_message = GENERATE_MISSING_INFO_QUERY_PROMPT
        human_message = f"""
        Original Topic: {original_query}\nMissing Information: {missing}
        
        CRITICAL: The following URLs previously failed to load or blocked our scraper. You MUST use the 'exclude_domains' parameter to avoid these sites: {failed_links}"""
        print(f"-> Supervisor: Executing Correction Search for: {missing}")
    else:
        system_message = GENERATE_SEARCH_QUERY_PROMPT
        human_message = f"Topic: {original_query}"
        print("-> Supervisor: Executing Initial Broad Search Strategy")

    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=human_message)
    ]
    
    # 2. Bind the schema directly to your Gemini LLM instance
    # This guarantees the model outputs a clean, typed Pydantic object
    structured_llm = llm.with_structured_output(SupervisorSearchSchema)
    
    # 3. Invoke the structured model
    structured_response = structured_llm.invoke(messages)
    
    # Convert Pydantic object directly to the dictionary your LangGraph expects
    # No json.loads() or string parsing required!
    search_queries = structured_response.model_dump()
    
    return {"search_queries": search_queries}

In [8]:
import importlib
# import RAG as RAG_module
# importlib.reload(RAG_module)
from RAG import chunks
from tools import tavily_search, extract_text
from vector_store import RDSVectorStore
from langchain_core.documents import Document as LangChainDoc

def web_and_ingestion_agent(state: GlobalState):
    """Executes searches, scrapes, chunks (LlamaIndex), and stores in Vector DB."""
    print("=== INGESTION AGENT: Searching and RAG processing...")

    queries = state['search_queries']

    links, response = tavily_search(queries['searches'], max_results=1)

    valid_docs = []
    failed_urls = []

    for link in links:
        extracted_data = extract_text(link)
        if extracted_data and extracted_data.get('page_content'):
            doc = LangChainDoc(
                page_content=extracted_data['page_content'],
                metadata=extracted_data['metadata']
            )
            valid_docs.append(doc)
        else:
            failed_urls.append(link)

    if not valid_docs:
        print("-> No valid documents extracted. Routing back.")
        return {
            "gathered_chunks": [], 
            "failed_urls": failed_urls,
            "missing_information": []
        }
    
    rds_wrapper = RDSVectorStore(collection_name='research_paper')
    rds_wrapper.initialize_store(embedding_manager=embed_model)

    chunker = chunks(vector_store=rds_wrapper.vectorstore)
    retriever = chunker.split_texts()

    print(f"Pushing {len(valid_docs)} source documents through the chunking pipeline...")
    retriever.add_documents(valid_docs)
    print("documents chunked successfully...")

    return {
        "gathered_chunks": valid_docs, 
        "failed_urls": failed_urls,
        "missing_information": []
    }

In [9]:
import json
from langchain_core.messages import HumanMessage, SystemMessage
from retriever import advanced_hybrid_retrieval # Import your hybrid search

def synthesis_and_critique_agent(state: dict):
    print("\n=== SYNTHESIS AGENT: Drafting and Critiquing ===")
    
    original_query = state["original_query"]
    ingested_docs = state.get("gathered_chunks", [])
    
    # ---------------------------------------------------------
    # PHASE 1: RETRIEVAL & CONTEXT FORMATTING
    # ---------------------------------------------------------
    print("-> Pulling best context from AWS RDS...")
    best_chunks = advanced_hybrid_retrieval(original_query, ingested_docs,embedding_manager=embed_model)
    
    formatted_references = []
    for doc in best_chunks:
        formatted_references.append({
            "content": doc.page_content,
            "url": doc.metadata.get("url", "Unknown Source")
        })
        
    context_string = json.dumps(formatted_references, indent=2)
    
    # ---------------------------------------------------------
    # PHASE 2: DRAFTING THE REPORT
    # ---------------------------------------------------------
    print("-> Drafting report with citations...")
    draft_system_prompt = (
        "You are an expert technical researcher. Write a comprehensive report answering the user's query.\n"
        "You MUST cite your sources using the URLs provided in the JSON context.\n\n"
        f"CONTEXT:\n{context_string}"
    )
    
    draft_response = llm.invoke([
        SystemMessage(content=draft_system_prompt),
        HumanMessage(content=original_query)
    ])
    draft = draft_response.content
    
    # ---------------------------------------------------------
    # PHASE 3: CRITIQUING THE DRAFT
    # ---------------------------------------------------------
    print("-> Executing self-critique loop...")
    critique_system_prompt = (
        "You are a strict, objective AI reviewer. Analyze the provided research draft against the original query.\n"
        "Identify if any critical information requested in the query is missing from the draft.\n\n"
        "You MUST output ONLY a valid JSON object in this exact format. Do NOT use markdown code blocks.\n"
        "{\n"
        '  "status": "complete" | "incomplete",\n'
        '  "missing_information": ["specific missing fact 1"] // empty list if complete\n'
        "}"
    )
    
    critique_user_message = f"ORIGINAL QUERY: {original_query}\n\nDRAFT REPORT:\n{draft}"
    
    critique_response = llm.invoke([
        SystemMessage(content=critique_system_prompt),
        HumanMessage(content=critique_user_message)
    ])
    
    # ---------------------------------------------------------
    # PHASE 4: STATE UPDATING & ROUTING
    # ---------------------------------------------------------
    try:
        # Strip out any Markdown blocks just in case the LLM hallucinates them
        clean_json = critique_response.content.replace("```json", "").replace("```", "").strip()
        critique_result = json.loads(clean_json)
    except Exception as e:
        print(f"-> Critique JSON Parsing Error: {e}. Bypassing critique.")
        # Failsafe: If the JSON breaks, we assume it's complete rather than crashing the loop
        critique_result = {"status": "complete", "missing_information": []}
        
    # Check if we need to loop back to the Supervisor
    if critique_result.get("status") == "incomplete" and len(critique_result.get("missing_information", [])) > 0:
        print(f"-> Critique Failed. Missing Info: {critique_result['missing_information']}")
        return {
            "draft_report": draft,
            "missing_information": critique_result["missing_information"],
            "is_complete": False
        }
    else:
        print("-> Critique Passed. Research complete.")
        return {
            "draft_report": draft,
            "missing_information": [],
            "is_complete": True,
            "formatted_references":formatted_references
        }


In [10]:
def routing_logic(state: GlobalState) -> str:
    """Decides if the graph should loop back or finish."""
    if state.get("is_complete", False):
        print("-> Routing: Data is complete. Ending graph.")
        return "end"
    else:
        print(f"-> Routing: Missing {state['missing_information']}. Looping back to Supervisor.")
        return "continue"

In [11]:
workflow = StateGraph(GlobalState)

# Add nodes
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("ingestion", web_and_ingestion_agent)
workflow.add_node("synthesis", synthesis_and_critique_agent)

#  Add Entry point
workflow.set_entry_point("supervisor")

workflow.add_edge('supervisor',"ingestion")
workflow.add_edge('ingestion',"synthesis")

workflow.add_conditional_edges(
    "synthesis",
    routing_logic, 
    {"continue": "supervisor", # If routing_logic returns "continue", go to supervisor
    "end": END }
)

app = workflow.compile()

In [12]:
# Explicitly pass the correct structural string


initial_state = {
    "original_query": "write a detailed report on history, cause, effect, and solution of bihar flood.",
    "search_queries": {},
    "gathered_chunks": [],
    "failed_urls": [],
    "missing_information": [],
    "draft_report": "",
    "is_complete": False,
    "formatted_references":[]
}

final_state = app.invoke(initial_state)
print("\n=== FINAL REPORT ===")
print(final_state["draft_report"])

=== Supervisor Agent working...
-> Supervisor: Executing Initial Broad Search Strategy
=== INGESTION AGENT: Searching and RAG processing...
=> Using Tavily to Search Internet for relevant information....

=> Executing Targeted search for Bihar floods history causes socio economic impact mitigation strategies disaster management from ['ndma.gov.in', 'sciencedirect.com', 'researchgate.net', 'reliefweb.int'] ....


c:\Users\krdhi\OneDrive\Desktop\RAG_tutorials\A_ARA\tools.py:40: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=max_results, search_depth="advanced", include_domains=tgt_domains,exclude_domains=exclude_domains)


=> Extracting texts from the https://www.sciencedirect.com/science/article/abs/pii/S2212420922005878....
=>extract_html in action....
Connecting to AWS RDS pgvector...


c:\Users\krdhi\OneDrive\Desktop\RAG_tutorials\A_ARA\vector_store.py:40: LangChainPendingDeprecationWarning: This class is pending deprecation and may be removed in a future version. You can swap to using the `PGVector` implementation in `langchain_postgres`. Please read the guidelines in the doc-string of this class to follow prior to migrating as there are some differences between the implementations. See <https://github.com/langchain-ai/langchain-postgres> for details about the new implementation.
  self.vectorstore = PGVector(


Pushing 1 source documents through the chunking pipeline...
documents chunked successfully...

=== SYNTHESIS AGENT: Drafting and Critiquing ===
-> Pulling best context from AWS RDS...
=== EXECUTING HYBRID RETRIEVAL & RERANKING FOR: 'write a detailed report on history, cause, effect, and solution of bihar flood.' ===
Connecting to AWS RDS pgvector...
-> Initializing Cross-Encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4079.83it/s]


-> Fetching, Fusing, and Reranking documents...
-> Drafting report with citations...
-> Executing self-critique loop...
-> Critique JSON Parsing Error: 'list' object has no attribute 'replace'. Bypassing critique.
-> Critique Passed. Research complete.
-> Routing: Data is complete. Ending graph.

=== FINAL REPORT ===
[{'type': 'text', 'text': '# Comprehensive Report on Bihar Floods: History, Causes, Effects, and Solutions\n\n---\n\n## Executive Summary\nBihar is India’s most flood-prone state, with North Bihar being particularly vulnerable. Approximately 73% of Bihar’s total geographical area (68,800 square kilometers out of 94,163 square kilometers) is susceptible to flooding. The state accounts for nearly 16.5% of India’s flood-affected area and over 22% of its flood-affected population. \n\nThe flood crisis in Bihar is a complex interplay of steep Himalayan topography, transboundary river dynamics with Nepal, heavy monsoon precipitation, high sediment loads, and historical policy sh

In [13]:
final_report = final_state["draft_report"][0]['text']

In [14]:
from markdown_pdf import MarkdownPdf, Section

pdf = MarkdownPdf(toc_level=2)
pdf.add_section(Section(final_report))

from pathlib import Path

base_dir = Path.cwd().parent
report_folder = base_dir / "reports"

report_folder.mkdir(parents=True, exist_ok=True)
filename = "my_report1.pdf"
# file_path = report_folder / filename
file_path = report_folder / filename

pdf.save(str(file_path))
print(f"Report saved successfully at: {file_path}")

Report saved successfully at: c:\Users\krdhi\OneDrive\Desktop\RAG_tutorials\reports\my_report1.pdf


In [15]:
final_state

{'original_query': 'write a detailed report on history, cause, effect, and solution of bihar flood.',
 'search_queries': {'searches': [{'query': 'Bihar floods history causes socio economic impact mitigation strategies disaster management',
    'target_domains': ['ndma.gov.in',
     'sciencedirect.com',
     'researchgate.net',
     'reliefweb.int'],
    'exclude_domains': ['quora.com',
     'pinterest.com',
     'reddit.com',
     'wikipedia.org']}]},
 'gathered_chunks': [Document(metadata={'url': 'https://www.sciencedirect.com/science/article/abs/pii/S2212420922005878'}, page_content='There was a problem providing the content you requested Please contact our support team for more information and provide the details below. Reference number: 9ff47303a9dd7e2c IP Address: 106.192.176.205 CPE00001 ::CLOUDFLARE_ERROR_1000S_BOX::')],
 'failed_urls': [],
 'missing_information': [],
 'draft_report': [{'type': 'text',
   'text': '# Comprehensive Report on Bihar Floods: History, Causes, Effects,